# CNN Autoencoder: Handwritten Digit Reconstruction (0–9)

This notebook demonstrates how to load the trained PyTorch CNN autoencoder, evaluate reconstruction quality on test samples from EMNIST Digits (0–9), and inspect the 64-dimensional latent representations.

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import datasets, transforms

# Add model path to sys.path
sys.path.append(os.path.abspath(os.path.join("..", "model")))
from autoencoder import ConvAutoencoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Load Trained Autoencoder Model

In [ ]:
model_path = os.path.join("..", "model", "saved_model.pt")
model = ConvAutoencoder(latent_dim=64).to(device)
model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
model.eval()
print(f"Loaded model weights from: {model_path}")

## 2. Load EMNIST Test Samples

In [ ]:
data_dir = os.path.join("..", "data")
transform = transforms.Compose([
    transforms.Lambda(lambda img: img.transpose(Image.TRANSPOSE)),
    transforms.ToTensor()
])

test_dataset = datasets.EMNIST(root=data_dir, split="digits", train=False, download=False, transform=transform)
print(f"Total test images: {len(test_dataset)}")

## 3. Compare Original vs. Reconstructed Digits

In [ ]:
num_samples = 8
indices = np.random.choice(len(test_dataset), num_samples, replace=False)

fig, axes = plt.subplots(2, num_samples, figsize=(14, 4))

with torch.no_grad():
    for i, idx in enumerate(indices):
        img, _ = test_dataset[idx]
        recon, z = model(img.unsqueeze(0).to(device))
        
        # Original
        axes[0, i].imshow(img.squeeze().numpy(), cmap="gray")
        axes[0, i].set_title("Original", fontsize=10)
        axes[0, i].axis("off")
        
        # Reconstructed
        axes[1, i].imshow(recon.squeeze().cpu().numpy(), cmap="gray")
        axes[1, i].set_title("Recon", fontsize=10)
        axes[1, i].axis("off")

plt.suptitle("CNN Autoencoder: Original vs. Reconstructed Character", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Latent Space Linear Interpolation (Morphing)

In [ ]:
idx_a, idx_b = indices[0], indices[1]
img_a, _ = test_dataset[idx_a]
img_b, _ = test_dataset[idx_b]

with torch.no_grad():
    za = model.encode(img_a.unsqueeze(0).to(device))
    zb = model.encode(img_b.unsqueeze(0).to(device))

steps = 9
fig, axes = plt.subplots(1, steps, figsize=(16, 2.5))

with torch.no_grad():
    for step, alpha in enumerate(np.linspace(0, 1, steps)):
        z_interp = (1.0 - alpha) * za + alpha * zb
        recon = model.decode(z_interp)
        axes[step].imshow(recon.squeeze().cpu().numpy(), cmap="gray")
        axes[step].set_title(f"alpha={alpha:.2f}", fontsize=10)
        axes[step].axis("off")

plt.suptitle("Latent Space Linear Trajectory: Morphing Character A into Character B", fontsize=13)
plt.tight_layout()
plt.show()